# Clustering Evaluation — Glomeruli Project (FP03)

Evaluates the cluster labels produced by the clustering pipeline (`cluster_hdbscan.py`,
`cluster_leiden.py`). It **loads the saved outputs** — it does not re-run clustering, and it
does **not** re-implement things the pipeline already does.

### What the pipeline already does (so this notebook does NOT repeat it)
- Preprocessing **L2 → PCA(0.9) → L2**, then UMAP + HDBSCAN/Leiden.
- **50-seed consensus** (`consensus_clustering`) → the saved labels are already the *stable*
  partition; unstable points are already marked noise. **Stability is handled** → no multi-seed
  section here.
- **Model selection** via a composite score (DBCV/modularity + intra-cluster cosine + penalties).
- Per-cluster **internal metrics** are already in `cluster_metrics.csv` → we load, not recompute.

### What this notebook adds (the report-relevant gaps)
1. **Consolidate** the per-cluster internal metrics (`cluster_metrics.csv`).
2. **Confound** — do clusters map onto the source slide (batch) instead of biology?
3. **Meaning vs severity** — do clusters order along the morphological gravity proxy? (Kruskal–Wallis)
4. **HDBSCAN (baseline) vs Leiden (extension)** — agreement (ARI/NMI) + the key test: where
   Leiden splits an HDBSCAN cluster, do the subgroups differ in severity (fully-sclerotic vs
   outline-full-but-still-structured)?
5. **Image grids** annotated by cluster.

### Input: each method's run folder `clustering results/<timestamp>/` (has `manifest.csv` +
`cluster_metrics.csv`). Set the two paths in Setup, then run top to bottom.

## 1. Setup — paths, severity proxy, slide ids

In [ ]:
import sys, glob
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / "src").is_dir():
    pass
elif (PROJECT_ROOT.parent / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

EMB_DIR   = PROJECT_ROOT / "data" / "glomeruli" / "embeddings"
CROPS_DIR = PROJECT_ROOT / "data" / "glomeruli" / "crops"
EVAL_DIR  = PROJECT_ROOT / "results" / "evaluation_notebook"   # morphology_scores.csv
OUT_DIR   = PROJECT_ROOT / "results" / "clustering_evaluation"
(OUT_DIR / "grids").mkdir(parents=True, exist_ok=True)

BACKBONE     = "mobilenet"
RANDOM_STATE = 42

# ---- POINT THESE AT YOUR TWO RUN FOLDERS (each holds manifest.csv + cluster_metrics.csv) ----
# e.g. PROJECT_ROOT / "clustering results" / "20260115_101500"
RUN_DIRS = {
    "HDBSCAN": None,   # <-- set to the HDBSCAN run folder
    "Leiden":  None,   # <-- set to the Leiden  run folder
}

# helper: list available run folders so you can pick the right timestamps
_cr = PROJECT_ROOT / "clustering results"
if _cr.is_dir():
    print("Available run folders in 'clustering results/':")
    for d in sorted(p for p in _cr.iterdir() if p.is_dir()):
        has = (d / "manifest.csv").exists()
        print(f"  {d.name}   {'[manifest.csv OK]' if has else '[no manifest]'}")
else:
    print("[INFO] 'clustering results/' not found — run cluster_hdbscan.py / cluster_leiden.py first,")
    print("       then set RUN_DIRS to the produced folders.")

In [ ]:
from sklearn.preprocessing import normalize
from sklearn.decomposition import PCA

def preprocess(emb):
    """Same preprocessing as the clustering pipeline: L2 -> PCA(0.9) -> L2."""
    l2 = normalize(emb, norm="l2")
    pca = PCA(n_components=0.9).fit(l2)
    return normalize(pca.transform(l2), norm="l2")

def slide_id_from_filename(fname):
    # RECHERCHE-016_0039.png -> RECHERCHE-016 (prefix before last underscore). ADAPT if needed.
    stem = Path(fname).stem
    return stem.rsplit("_", 1)[0] if "_" in stem else stem

# severity proxy (from evaluation.ipynb Section 5); reference crop order = its 'crop' column
scores_csv = EVAL_DIR / "morphology_scores.csv"
if scores_csv.exists():
    sdf = pd.read_csv(scores_csv)
    ref_crops = sdf["crop"].astype(str).tolist()
    morphology_score = sdf["morphology_score"].to_numpy(dtype=float)
    print(f"Severity proxy loaded for {len(ref_crops)} crops.")
else:
    ref_crops = sorted(p.name for p in CROPS_DIR.iterdir() if p.suffix.lower()==".png")
    morphology_score = None
    print("[WARN] morphology_scores.csv not found -> severity sections skipped.")

crop_to_ref = {c: i for i, c in enumerate(ref_crops)}
slide_ids = np.array([slide_id_from_filename(c) for c in ref_crops])
print("Distinct slides:", len(set(slide_ids.tolist())))

# embeddings (optional, for cross-check silhouette) preprocessed like the pipeline
emb_hits = sorted(EMB_DIR.glob(f"{BACKBONE}*.npy"))
Xp = preprocess(np.load(emb_hits[0]).astype(np.float64)) if emb_hits else None
print("Embeddings (preprocessed):", None if Xp is None else Xp.shape)

## 2. Load the labels (`manifest.csv`) and the pre-computed metrics (`cluster_metrics.csv`)

`manifest.csv` columns: `id, cluster, image_path`. We align each label vector to the reference
crop order **by image filename**. `-1` = noise, `-2` = crop not present in that run.

In [ ]:
def load_manifest(run_dir, ref):
    run_dir = Path(run_dir)
    man = pd.read_csv(run_dir / "manifest.csv")
    lab = np.full(len(ref), -2, dtype=int)
    for _, r in man.iterrows():
        base = Path(str(r["image_path"])).name
        if base in crop_to_ref:
            lab[crop_to_ref[base]] = int(r["cluster"])
    matched = int(np.sum(lab != -2))
    print(f"  {run_dir.name}: matched {matched}/{len(ref)} crops")
    metrics = None
    mp = run_dir / "cluster_metrics.csv"
    if mp.exists(): metrics = pd.read_csv(mp)
    return lab, metrics

labels, metrics = {}, {}
for name, d in RUN_DIRS.items():
    if d is None:
        print(f"[skip] {name}: RUN_DIRS['{name}'] not set")
        continue
    labels[name], metrics[name] = load_manifest(d, ref_crops)

def summarize(lab, name):
    v = lab[lab != -2]
    clusters = sorted(set(int(x) for x in np.unique(v) if x != -1))
    noise = int(np.sum(v == -1))
    print(f"{name}: {len(clusters)} clusters, {noise} noise "
          f"({100*noise/max(len(v),1):.1f}%), {int(np.sum(lab==-2))} missing")

for name, lab in labels.items():
    summarize(lab, name)

## 3. Internal metrics per cluster *(loaded from the pipeline)*

DBCV / modularity, size and intra-cluster cosine are already produced during model selection —
we just display them. A modest silhouette / negative DBCV is expected (continuum) and confirms
the *coarse partition* reading rather than condemning the partition.

In [ ]:
from sklearn.metrics import silhouette_score

for name, lab in labels.items():
    print(f"=== {name}: cluster_metrics.csv (from the pipeline) ===")
    if metrics.get(name) is not None:
        display(metrics[name])
    # optional cross-check: overall silhouette on the pipeline space (non-noise)
    if Xp is not None:
        m = (lab != -2)
        nn = lab[m] != -1
        if len(set(lab[m][nn].tolist())) >= 2:
            try:
                s = float(silhouette_score(Xp[m][nn], lab[m][nn]))
                print(f"   [cross-check] overall silhouette (non-noise) = {s:.3f}")
            except Exception as e:
                print("   silhouette n/a:", e)

## 4. Confound — clusters vs source slide

A cluster dominated by one slide is a batch/stain artefact, not biology.

In [ ]:
def norm_entropy(counts):
    a = np.asarray(counts, float); tot = a.sum()
    if tot <= 0: return np.nan
    p = a[a > 0]/tot
    return float(-(np.sum(p*np.log(p)))/np.log(len(a))) if len(a) > 1 else 0.0

for name, lab in labels.items():
    m = lab != -2
    ct = pd.crosstab(pd.Series(lab[m], name="cluster"), pd.Series(slide_ids[m], name="slide"))
    rows = []
    for cl, row in ct.iterrows():
        tot = int(row.sum())
        rows.append({"cluster": cl, "n": tot,
                     "dominant_slide": str(row.idxmax()),
                     "dominant_slide_frac": round(row.max()/tot, 3) if tot else np.nan,
                     "slide_entropy": round(norm_entropy(row.values), 3),
                     "n_slides": int((row > 0).sum())})
    sc = pd.DataFrame(rows)
    sc.to_csv(OUT_DIR / f"confound_slide_{name}.csv", index=False)
    print(f"=== {name}: clusters vs slide (dominant_slide_frac ~1 = artefact) ==="); display(sc)

## 5. Meaning — do clusters order along the severity proxy?

If the clusters are a partition of the severity continuum, the morphological proxy should differ
across clusters and they should order along it. Boxplot per cluster + **Kruskal–Wallis**.

In [ ]:
from scipy.stats import kruskal

if morphology_score is not None:
    for name, lab in labels.items():
        m = (lab != -2) & (lab != -1)
        clusters = sorted(set(lab[m].tolist()))
        groups = [morphology_score[m & (lab == c)] for c in clusters]
        groups = [g for g in groups if len(g) >= 3]
        med = pd.DataFrame({"cluster": clusters,
                            "median_severity": [float(np.median(morphology_score[m & (lab==c)])) for c in clusters],
                            "n": [int(np.sum(m & (lab==c))) for c in clusters]}).sort_values("median_severity")
        med.to_csv(OUT_DIR / f"severity_by_cluster_{name}.csv", index=False)
        print(f"=== {name} ===")
        if len(groups) >= 2:
            kw = kruskal(*groups)
            print(f"Kruskal-Wallis: H={kw.statistic:.1f}, p={kw.pvalue:.2e} "
                  f"({'clusters differ on severity' if kw.pvalue < 0.05 else 'no significant difference'})")
        display(med)
        order = med["cluster"].tolist()
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.boxplot([morphology_score[m & (lab==c)] for c in order], labels=[str(c) for c in order], showfliers=False)
        ax.set_title(f"{name}: severity proxy by cluster (ordered by median)")
        ax.set_xlabel("cluster"); ax.set_ylabel("morphology severity (proxy)")
        ax.spines[["top","right"]].set_visible(False)
        fig.tight_layout(); fig.savefig(OUT_DIR / f"severity_by_cluster_{name}.png", dpi=150, bbox_inches="tight"); plt.show()
else:
    print("No morphology_score -> skipping (run evaluation.ipynb Section 5 first).")

## 6. HDBSCAN (baseline) vs Leiden (extension)

Agreement (ARI/NMI) + the claimed advantage: where **Leiden splits an HDBSCAN cluster into ≥2
subgroups**, do those subgroups differ on severity? A significant gap = Leiden separates
fully-sclerotic from outline-full-but-still-structured glomeruli (the extension's value).

In [ ]:
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from scipy.stats import mannwhitneyu

if {"HDBSCAN","Leiden"}.issubset(labels):
    h, l = labels["HDBSCAN"], labels["Leiden"]
    both = (h != -2) & (l != -2) & (h != -1) & (l != -1)
    print(f"Common non-noise points: {int(both.sum())}")
    print(f"ARI = {adjusted_rand_score(h[both], l[both]):.3f} | NMI = {normalized_mutual_info_score(h[both], l[both]):.3f}")
    ct = pd.crosstab(pd.Series(h[both], name="HDBSCAN"), pd.Series(l[both], name="Leiden"))
    ct.to_csv(OUT_DIR / "hdbscan_vs_leiden_crosstab.csv")
    print("Crosstab (rows=HDBSCAN, cols=Leiden): a HDBSCAN row spread over >1 Leiden col = a split")
    display(ct)

    if morphology_score is not None:
        rows = []
        for hc in sorted(set(h[both].tolist())):
            sub = both & (h == hc)
            leid = l[sub]
            subs = [c for c in sorted(set(leid.tolist())) if np.sum(leid == c) >= 5]
            if len(subs) >= 2:
                meds = {c: float(np.median(morphology_score[sub & (l == c)])) for c in subs}
                lo, hi = min(meds, key=meds.get), max(meds, key=meds.get)
                a, b = morphology_score[sub & (l == lo)], morphology_score[sub & (l == hi)]
                try: p = float(mannwhitneyu(a, b, alternative="two-sided").pvalue)
                except Exception: p = np.nan
                rows.append({"hdbscan_cluster": hc, "n": int(sub.sum()), "n_leiden_subclusters": len(subs),
                             "median_sev_low": round(meds[lo],3), "median_sev_high": round(meds[hi],3),
                             "severity_gap": round(meds[hi]-meds[lo],3), "mannwhitney_p": p})
        split = pd.DataFrame(rows)
        if not split.empty:
            split.to_csv(OUT_DIR / "leiden_splits_vs_severity.csv", index=False)
            print("\nWhere Leiden splits a HDBSCAN cluster: do the subgroups differ on severity?")
            print("(significant p + a severity_gap = Leiden's finer separation is morphological)")
            display(split)
        else:
            print("\nLeiden does not split any HDBSCAN cluster into >=2 subgroups here.")
else:
    print("Need BOTH HDBSCAN and Leiden runs set in RUN_DIRS for this comparison.")

## 7. Image grids per cluster *(qualitative)*

Contact sheet per cluster per method (complements the pipeline's `samples_per_cluster.png`).

In [ ]:
from PIL import Image, ImageOps, ImageDraw
import math
MAX_PER_CLUSTER, COLS, TILE = 24, 6, 140

def grid_for(indices, title, out_path):
    paths = [CROPS_DIR / ref_crops[i] for i in indices][:MAX_PER_CLUSTER]
    if not paths: return
    rows = math.ceil(len(paths)/COLS)
    canvas = Image.new("RGB", (COLS*TILE, rows*TILE+22), (245,245,245))
    ImageDraw.Draw(canvas).text((6,6), title, fill=(0,0,0))
    for k, p in enumerate(paths):
        try: im = ImageOps.contain(Image.open(p).convert("RGB"), (TILE,TILE))
        except Exception: im = Image.new("RGB",(TILE,TILE),(230,230,230))
        c = Image.new("RGB",(TILE,TILE),(255,255,255)); c.paste(im,((TILE-im.width)//2,(TILE-im.height)//2))
        canvas.paste(c, ((k%COLS)*TILE, 22+(k//COLS)*TILE))
    canvas.save(out_path)

if CROPS_DIR.is_dir():
    rng = np.random.default_rng(RANDOM_STATE)
    for name, lab in labels.items():
        d = OUT_DIR / "grids" / name; d.mkdir(parents=True, exist_ok=True)
        for cl in sorted(set(int(x) for x in lab if x not in (-1, -2))):
            idx = np.where(lab == cl)[0]
            if len(idx) > MAX_PER_CLUSTER: idx = rng.choice(idx, MAX_PER_CLUSTER, replace=False)
            grid_for(sorted(idx.tolist()), f"{name} | cluster {cl} (n={int(np.sum(lab==cl))})", d / f"cluster_{cl}.png")
        print(f"{name}: grids saved in {d}")
else:
    print("No crops dir -> skipping grids.")

## 8. How to read it (for the report)

- **Internal metrics**: modest silhouette / DBCV ≤ 0 → confirms *coarse partition of a continuum*.
- **Confound**: `dominant_slide_frac` well below 1 → clusters are not slide artefacts.
- **Severity**: significant Kruskal–Wallis + clusters ordered by median proxy → the partition
  reflects the gravity gradient.
- **HDBSCAN vs Leiden**: a significant `severity_gap` between Leiden subgroups inside one HDBSCAN
  cluster is the quantitative evidence that **Leiden (extension) separates fully-sclerotic from
  still-structured glomeruli**, which HDBSCAN (baseline) merges.

Stability is not evaluated here on purpose: it is already handled by the 50-seed
`consensus_clustering` upstream. Outputs are under `results/clustering_evaluation/`.